# SparkSession and your first DataFrame

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").appName("learning-spark").getOrCreate()

my_range = spark.range(1000).toDF("number")
my_range.printSchema()
print(my_range.rdd.getNumPartitions())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 08:57:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- number: long (nullable = false)

16


* Partition: a chunk of rows that one task processes. 16 here, because local[*] uses every core on this machine.
* DataFrames are immutable. Every operation returns a new DataFrame; nothing is modified in place.

Common mistake

Partitions are the unit of parallelism. One partition means one task at a time, no matter how many cores or executors you have. Too many tiny partitions waste time on scheduling.

In [5]:
my_range

DataFrame[number: bigint]

In [6]:
div_by_2 = my_range.where("number % 2 = 0")   # transformation: no job, no output
div_by_2

DataFrame[number: bigint]

In [13]:
div_by_2.count()                               # action: runs a job (this is execution)

500

In [15]:
div_by_2.explain()  # THis is not an execution !

== Physical Plan ==
*(1) Project [id#0L AS number#1L]
+- *(1) Filter ((id#0L % 2) = 0)
   +- *(1) Range (0, 1000, step=1, splits=16)




In [16]:
div_by_2.printSchema()  # THis is not an execution !

root
 |-- number: long (nullable = false)

